# Lab 5 - Exploring the Landmark Data

The goal of this notebook is to understand exactly what MediaPipe returns for hands, pose, and face landmarks. We use static reference images so the data stays frozen while you inspect it across multiple cells.

Division of labour in Lab 5:

- This notebook = understand the data: static, exploratory, read-and-answer.
- `familiarization/*.py` scripts = see it run live with a webcam.
- `challenges/*` = use the data to build something.

No webcam needed. Run every cell top to bottom and read the printed output.


## MediaPipe references

This notebook uses MediaPipe's older `mp.solutions` API. These references match the objects used below:

- [MediaPipe Python package setup](https://ai.google.dev/edge/mediapipe/solutions/setup_python)
- [Hands: Python Solution API](https://github.com/google-ai-edge/mediapipe/blob/master/docs/solutions/hands.md#python-solution-api) - `mp.solutions.hands.Hands`, `multi_hand_landmarks`, `multi_handedness`, `HAND_CONNECTIONS`
- [Pose: Python Solution API](https://github.com/google-ai-edge/mediapipe/blob/master/docs/solutions/pose.md#python-solution-api) - `mp.solutions.pose.Pose`, `pose_landmarks`, `visibility`, `POSE_CONNECTIONS`
- [Face Mesh: Python Solution API](https://github.com/google-ai-edge/mediapipe/blob/master/docs/solutions/face_mesh.md#python-solution-api) - `mp.solutions.face_mesh.FaceMesh`, `multi_face_landmarks`, `refine_landmarks`
- [Landmark proto fields](https://github.com/google-ai-edge/mediapipe/blob/master/mediapipe/framework/formats/landmark.proto) - `x`, `y`, `z`, `visibility`, and `presence`


### Current MediaPipe Tasks API equivalents

MediaPipe's current docs increasingly use `mp.tasks.vision`. The concepts are similar, but object names and result structures differ from the `mp.solutions` code in this notebook:

- [Hand Landmarker for Python](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python)
- [Pose Landmarker for Python](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker/python)
- [Face Landmarker for Python](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python)


In [ ]:
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd()
REF_DIR_CANDIDATES = [
    cwd / "ref_imgs",
    cwd.parent / "ref_imgs",
    cwd / "Lab_5" / "ref_imgs",
    cwd / "AppCV_2026" / "Lab_5" / "ref_imgs",
]
REF_DIR = next((path for path in REF_DIR_CANDIDATES if path.exists()), None)
assert REF_DIR is not None, "Could not find ref_imgs. Start Jupyter from Lab_5, familiarization, or the course repo root."
print(f"Using reference images from: {REF_DIR}")


In [ ]:
def show(img_bgr, title=None):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 4))
    plt.imshow(img_rgb)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


OpenCV loads color images as **BGR**, while matplotlib expects **RGB**. The `show()` helper converts before display.

In [ ]:
IMAGE_FILES = {
    "hands": "hands.jpg",
    "body": "body.jpg",
    "face": "face.jpg",
}

images = {}
for key, filename in IMAGE_FILES.items():
    path = REF_DIR / filename
    img = cv2.imread(str(path))
    assert img is not None, (
        f"OpenCV could not load {path}. Check that it exists and is a true JPEG/PNG file, "
        "not another format with a .jpg extension."
    )
    images[key] = img
    print(f"{filename}: {img.shape}")


Note the shape of each image: `(H, W, 3)`. The `3` is the BGR channels. Hold onto `H` and `W`. We will use them to turn landmark coordinates into pixels.


In [ ]:
for key in ["hands", "body", "face"]:
    show(images[key], IMAGE_FILES[key])


## Hands: 21 landmarks

`mp.solutions.hands` detects one or more hands. Its main output is `multi_hand_landmarks`: one landmark set per detected hand. It also provides `multi_handedness`, which classifies each detected hand as left or right.


In [ ]:
mp_hands = mp.solutions.hands

with mp_hands.Hands(
    static_image_mode=True,  # Still image: detect from scratch instead of tracking a video stream.
    max_num_hands=2,
    min_detection_confidence=0.5,
) as hands_model:
    hand_results = hands_model.process(cv2.cvtColor(images["hands"], cv2.COLOR_BGR2RGB))


In [ ]:
hand_landmarks = hand_results.multi_hand_landmarks
print("type(results.multi_hand_landmarks):", type(hand_landmarks))
print("number of detected hands:", 0 if hand_landmarks is None else len(hand_landmarks))


How many hands were detected? What kind of object is `multi_hand_landmarks`, and what do you think it holds if no hand is found?


In [ ]:
assert hand_landmarks is not None, "No hands were detected in hands.jpg. Check the reference image."
hand = hand_landmarks[0]
print("landmarks in first hand:", len(hand.landmark))


In [ ]:
mp_drawing = mp.solutions.drawing_utils

for hand_idx, detected_hand in enumerate(hand_landmarks):
    handedness = hand_results.multi_handedness[hand_idx].classification[0]
    print(f"Hand {hand_idx} ({handedness.label}, score={handedness.score:.3f})")
    for idx, landmark in enumerate(detected_hand.landmark):
        print(f"  {idx:>2}: x={landmark.x:.3f}, y={landmark.y:.3f}, z={landmark.z:.3f}")

hand_skeleton = images["hands"].copy()
for detected_hand in hand_landmarks:
    mp_drawing.draw_landmarks(
        hand_skeleton,
        detected_hand,
        mp_hands.HAND_CONNECTIONS,
        landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=2),
        connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 160, 0), thickness=1),
    )

show(hand_skeleton, "Hand skeleton")


In [ ]:
index_tip = hand.landmark[8]
print(index_tip)
print("x:", index_tip.x)
print("y:", index_tip.y)
print("z:", index_tip.z)


What three fields does each landmark have?


In [ ]:
xs = np.array([landmark.x for landmark in hand.landmark])
ys = np.array([landmark.y for landmark in hand.landmark])
print(f"x min/max: {xs.min():.3f} to {xs.max():.3f}")
print(f"y min/max: {ys.min():.3f} to {ys.max():.3f}")


What range do `x` and `y` fall in? Are these pixels or something else?

The `z` value is relative depth. Treat it as approximate; smaller values are closer to the camera.


In [ ]:
hand_pixels = images["hands"].copy()
h, w, _ = hand_pixels.shape
points_to_draw = {
    0: "0 wrist",
    4: "4 thumb tip",
    8: "8 index tip",
}

for idx, label in points_to_draw.items():
    landmark = hand.landmark[idx]
    px = int(landmark.x * w)
    py = int(landmark.y * h)
    cv2.circle(hand_pixels, (px, py), 4, (0, 0, 255), -1)
    cv2.putText(hand_pixels, label, (px + 5, py - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    print(f"{label}: normalized=({landmark.x:.3f}, {landmark.y:.3f}), pixels=({px}, {py})")

show(hand_pixels, "Selected hand landmarks")


This `x * W`, `y * H` conversion is exactly what the challenge GUIs do to draw the red circles for the keypoints you are using. You now understand that debugging tool.


In [ ]:
handedness = hand_results.multi_handedness[0].classification[0]
print("label:", handedness.label)
print("score:", handedness.score)


How does MediaPipe report left vs right, and how confident is it? Which challenge will need this?


In [ ]:
POINT_TO_DRAW = 8  # Try 12, 16, or 20, then re-run this cell.

exercise_img = images["hands"].copy()
h, w, _ = exercise_img.shape
landmark = hand.landmark[POINT_TO_DRAW]
px = int(landmark.x * w)
py = int(landmark.y * h)
cv2.circle(exercise_img, (px, py), 5, (0, 0, 255), -1)
cv2.putText(exercise_img, str(POINT_TO_DRAW), (px + 6, py - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
show(exercise_img, f"Hand landmark {POINT_TO_DRAW}")


## Pose / body: 33 landmarks, plus visibility

`mp.solutions.pose` returns a single `pose_landmarks` object for the person it detects. Unlike hands and faces, this is not a list of multiple people.


In [ ]:
mp_pose = mp.solutions.pose

with mp_pose.Pose(
    static_image_mode=True,  # Still image: detect from scratch instead of tracking a video stream.
    model_complexity=1,
    min_detection_confidence=0.5,
) as pose_model:
    pose_results = pose_model.process(cv2.cvtColor(images["body"], cv2.COLOR_BGR2RGB))

if pose_results.pose_landmarks is None:
    pose_points = []
    print("Pose not detected on body.jpg. Try a brighter, clearer full-body reference image.")
else:
    pose_points = pose_results.pose_landmarks.landmark
    print("Pose detected on body.jpg.")


In [ ]:
if pose_points:
    right_wrist = pose_points[16]
    print("pose landmark count:", len(pose_points))
    print(right_wrist)
    print("x:", right_wrist.x)
    print("y:", right_wrist.y)
    print("z:", right_wrist.z)
    print("visibility:", right_wrist.visibility)
else:
    print("Skipping landmark inspection because no pose was detected.")


In [ ]:
if pose_points:
    for idx, landmark in enumerate(pose_points):
        print(
            f"{idx:>2}: x={landmark.x:.3f}, y={landmark.y:.3f}, "
            f"z={landmark.z:.3f}, visibility={landmark.visibility:.3f}"
        )

    pose_skeleton = images["body"].copy()
    mp_drawing.draw_landmarks(
        pose_skeleton,
        pose_results.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=2),
        connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 160, 0), thickness=1),
    )
    show(pose_skeleton, "Pose skeleton")
else:
    print("Skipping full landmark print and skeleton because no pose was detected.")


What new field does a pose landmark have that a hand landmark did not? What might `visibility` near 0 mean?


In [ ]:
if pose_points:
    visibility_samples = {
        0: "nose",
        16: "right wrist",
        27: "left ankle",
        28: "right ankle",
    }
    for idx, label in visibility_samples.items():
        print(f"{idx:>2} {label:>12}: visibility={pose_points[idx].visibility:.3f}")
else:
    print("Skipping visibility inspection because no pose was detected.")


The jumping-jack challenge should check `visibility` before trusting a point. Otherwise an off-screen or low-confidence ankle can give unreliable coordinates.


In [ ]:
pose_img = images["body"].copy()
h, w, _ = pose_img.shape
pose_points_to_draw = {
    11: "11 L shoulder",
    12: "12 R shoulder",
    15: "15 L wrist",
    16: "16 R wrist",
    27: "27 L ankle",
    28: "28 R ankle",
}

if pose_points:
    for idx, label in pose_points_to_draw.items():
        landmark = pose_points[idx]
        px = int(landmark.x * w)
        py = int(landmark.y * h)
        cv2.circle(pose_img, (px, py), 3, (0, 0, 255), -1)
        cv2.putText(pose_img, label, (px + 4, py - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 255), 1)
    show(pose_img, "Selected pose landmarks")
else:
    show(pose_img, "body.jpg: no pose detected")


## Face: 468 landmarks, or 478 with refined landmarks

`mp.solutions.face_mesh` returns `multi_face_landmarks`: one landmark set per detected face. Do not panic about the number of points. It is the same structure you already inspected; you will usually use only a handful of indices.


In [ ]:
mp_face_mesh = mp.solutions.face_mesh

with mp_face_mesh.FaceMesh(
    static_image_mode=True,  # Still image: detect from scratch instead of tracking a video stream.
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
) as face_model:
    face_results = face_model.process(cv2.cvtColor(images["face"], cv2.COLOR_BGR2RGB))

face_landmarks = face_results.multi_face_landmarks
if face_landmarks is None:
    face_points = []
    print("No face detected on face.jpg. Check the reference image.")
else:
    face = face_landmarks[0]
    face_points = face.landmark
    print("number of detected faces:", len(face_landmarks))
    print("landmarks in first face:", len(face_points))
    print("sample landmark 61:")
    print(face_points[61])


In [ ]:
if face_points:
    for idx, landmark in enumerate(face_points):
        print(f"{idx:>3}: x={landmark.x:.3f}, y={landmark.y:.3f}, z={landmark.z:.3f}")
else:
    print("Skipping full landmark print because no face was detected.")


How many points now? Is a single landmark any different in structure from a hand landmark?


In [ ]:
face_img = images["face"].copy()
h, w, _ = face_img.shape
smile_points = {
    61: "61 mouth corner",
    291: "291 mouth corner",
    13: "13 upper lip",
    14: "14 lower lip",
}

if face_points:
    for idx, label in smile_points.items():
        landmark = face_points[idx]
        px = int(landmark.x * w)
        py = int(landmark.y * h)
        cv2.circle(face_img, (px, py), 3, (0, 0, 255), -1)
        cv2.putText(face_img, label, (px + 4, py - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.32, (0, 0, 255), 1)
    show(face_img, "Smile-relevant face landmarks")
else:
    show(face_img, "face.jpg: no face detected")


These are the exact indices the smile challenge suggests. Now you can see where they sit.

Notice that raw pixel distance between mouth corners changes when you move closer or farther from the camera. That is why robust detectors use **ratios**, which the landmark-reference material covers next.


## Recap and bridge to the challenges

| Model | Result attribute | # landmarks | Per-landmark fields | Multiple subjects? |
|-------|------------------|-------------|---------------------|--------------------|
| Hands | `multi_hand_landmarks` plus `multi_handedness` | 21 | x, y, z | list, one per hand |
| Pose | `pose_landmarks` | 33 | x, y, z, **visibility** | single |
| Face | `multi_face_landmarks` | 468, or 478 refined | x, y, z | list, one per face |

Three things carry into every challenge:

1. Landmark **indices** identify body parts.
2. Coordinates are **normalized 0-1**; multiply by image width and height for pixels.
3. Compare points by **ratios**, not raw distances, when you need camera-distance robustness.


### What each challenge function actually receives

The raw MediaPipe result is not always passed directly into your `detect_*()` function. Each challenge's `main.py` unpacks the data first:

| Challenge | Function | Argument shape |
|-----------|----------|----------------|
| `smile_detection` | `detect_emotion(face_landmarks)` | the flat `.landmark` list for one face |
| `jumping_jack_counter` | `detect_jumping_jack(pose_landmarks)` | the flat `.landmark` list of 33 pose points |
| `clapping_counter` | `detect_clap(hand_landmarks_list)` | a list of `.landmark` lists, one per hand, with no handedness |
| `number_recognition` | `count_fingers(hand_data_list)` | a list of `(landmark_list, "Left"/"Right")` tuples |
| `thumbs_decision` | `detect_thumbs_decision(hand_landmarks_list)` | a list of hand `.landmark` lists |

Match raw result to function argument by opening the relevant `main.py` file and finding the call into `detection_logic.py`. The clearest examples are:

- `challenges/number_recognition/main.py`: builds `(hand_landmarks.landmark, hand_classification)` tuples before calling `count_fingers(...)`.
- `challenges/clapping_counter/main.py`: builds `all_hand_landmarks` as a list of `.landmark` lists before calling `detect_clap(...)`.

Next: run the live `familiarization/*.py` scripts to see this in motion, then open `challenges/README.md`.
